# October 10 event analysis

Build the five-minute table with the repository SQL, then reproduce Figure 1. Run this notebook from the repository root.

In [ ]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

repo = Path.cwd()
con = duckdb.connect('jupiter_lend.duckdb')
sql = (repo / 'sql/analysis/create_liquidation_analysis_5m.sql').read_text()
con.execute(sql)
analysis = con.sql('SELECT * FROM liquidation_analysis_5m ORDER BY timestamp').df()
analysis.head()

In [ ]:
fig, debt_ax = plt.subplots(figsize=(13, 6))
price_ax = debt_ax.twinx()

debt_ax.bar(analysis['timestamp'], analysis['debt_repaid_usd'], width=0.003,
            color='#4666a6', alpha=0.75, label='Debt repaid')
price_ax.plot(analysis['timestamp'], analysis['sol_close'],
              color='#d97732', linewidth=1.8, label='SOL close')
debt_ax.set_ylabel('Debt repaid (USD)')
price_ax.set_ylabel('SOL price (USD)')
debt_ax.set_xlabel('UTC time, October 10, 2025')
debt_ax.set_title('Jupiter Lend debt repaid and SOL price in five-minute intervals')
debt_ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M', tz=mdates.UTC))
debt_ax.grid(axis='y', alpha=0.2)
handles_1, labels_1 = debt_ax.get_legend_handles_labels()
handles_2, labels_2 = price_ax.get_legend_handles_labels()
debt_ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc='upper left')
fig.tight_layout()
output = repo / 'docs/images/figure-1-debt-repaid-vs-sol-price.png'
fig.savefig(output, dpi=200, bbox_inches='tight')
plt.show()